In [9]:
#Importando bibliotecas
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy, Precision, Recall

In [11]:
#Baixando datasets
from torchvision import datasets
import torchvision.transforms as transforms

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 272kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.51MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 16.1MB/s]


In [12]:
#Analisando o número de classes possíveis nos dados
classes = train_data.classes
n_classes = len(train_data.classes)
print(classes)

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [13]:
#Definindo parâmetros para o modelo
n_input_channels = 1
n_output_channels = 16
image_size = train_data[0][0].shape[1]
batch_size = 10
n_epochs = 1

In [14]:
#Definindo nossa rede neural usando POO
class Net(nn.Module):
    def __init__(self, n_classes, n_input_channels, n_output_channels):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(n_input_channels, n_output_channels, kernel_size = 2, padding=1)
        self.elu = nn.ELU()
        self.maxpool = nn.MaxPool2d(kernel_size = 2)
        self.flat = nn.Flatten()
        self.clf = nn.Linear(n_output_channels * 14 * 14, n_classes)
    def forward(self, x):
        x = self.conv1(x)
        x = self.elu(x)
        x = self.maxpool(x)
        x = self.flat(x)
        x = self.clf(x)
        return x

In [15]:
#Definindo classe para treinar o modelo
def train_model(optimizer, net, n_epochs):
    n_processed = 0
    criterion = nn.CrossEntropyLoss()
    for epoch in range(n_epochs):
        running_loss = 0
        n_processed = 0
        for features, labels in dataloader_train:
            optimizer.zero_grad()
            outputs = net(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss = running_loss + loss.item()
            n_processed = n_processed + len(labels)
            #print(f'epoch: {epoch}')
            #print(f'loss: {running_loss/n_processed}')
    total_loss_train = running_loss / len(dataloader_train)
    print(f'Total loss: {total_loss_train}')

In [16]:
#Importando os dados de treino
dataloader_train = DataLoader(train_data, shuffle=True, batch_size = batch_size)

In [17]:
#Treinando o modelo
net = Net(n_classes, n_input_channels, n_output_channels)
optimizer = optim.Adam(net.parameters(), lr = 0.001)

train_model(optimizer = optimizer, net = net, n_epochs = n_epochs)

Total loss: 0.44041873192143005


In [ ]:
#Importando dados de teste, e definindo métricas
dataloader_test = DataLoader(test_data, batch_size = 10, shuffle = False)

accuracy_metric = Accuracy(task='multiclass', n_classes=n_classes)
precision_metric = Precision(task='multiclass', n_classes=n_classes, average=None)
recall_metric = Recall(task='multiclass', n_classes=n_classes, average=None)

In [ ]:
#Usando o modelo para fazer previsões
net.eval()
predictions = []
with torch.no_grad():
    for i, (features, labels) in enumerate(dataloader_test):
        output = net.forward(features.reshape(-1, 1, image_size, image_size))
        cat = torch.argmax(output, dim=-1)
        predictions.extend(cat.tolist())
        accuracy_metric(cat, labels)
        precision_metric(cat, labels)
        recall_metric(cat, labels)

In [ ]:
#Calculando métricas
accuracy = accuracy_metric.compute().item()
precision = precision_metric.compute().tolist()
recall = recall_metric.compute().tolist()

In [ ]:
#Exibindo métricas
print('Accuracy:', accuracy)
print("Recall per class: ")
dict_recall = dict(zip(classes, recall))
for key, value in dict_recall.items():
    print(f'{key}: {value} \n')
print("Precision per class: ")
dict_precision = dict(zip(classes, precision))
for key, value in dict_precision.items():
    print(f'{key}: {value} \n')